# rpascap — Stage 2 prototype pipeline (Colab)

**Status: prototype for Stage 2/3 verification. Nothing produced by this notebook is a manuscript result.**

Modes: `DEMO=True` uses only synthetic data (minutes). `DEMO=False` downloads the official pNEUMA archive (15.8 GB, Zenodo 10.5281/zenodo.10491409, CC BY 4.0) and Open-Meteo wind, verifies checksums, and builds the exposure and wind tables used in Stage 4.

In [ ]:
DEMO = True
!pip -q install -r requirements.txt
import sys, platform, scipy, numpy; print(sys.version, platform.platform(), scipy.__version__, numpy.__version__)

## 1. Tests (must pass before anything else)

In [ ]:
!python -m pytest -q

## 2. Official data (full mode only)
Required citation: Barmpounakis & Geroliminis (2020) doi:10.1016/j.trc.2019.11.023; dataset doi:10.5281/zenodo.10491409; acknowledgement 'Data source: pNEUMA – open-traffic.epfl.ch'.

In [ ]:
from pathlib import Path
from rpascap.data import download
LOG = Path('provenance_log.jsonl')
if not DEMO:
    zip_path = download.download_pneuma(Path('data/raw/pneuma'), LOG)   # md5 verified inside
    !unzip -q -n {zip_path} -d data/raw/pneuma/
    fc, ob = download.fetch_wind(37.98, 23.73, '2024-01-01', '2026-10-31', LOG)

## 3. Road-occupant exposure from pNEUMA (full mode)
Vehicle positions -> vehicle-seconds per 20 m cell -> occupants (assumed occupancy and sheltering, reported separately). This is NOT pedestrian or residential population.

In [ ]:
import glob, pandas as pd
from rpascap.data import pneuma
if not DEMO:
    files = sorted(glob.glob('data/raw/pneuma/**/*.csv', recursive=True))
    grid = pneuma.Grid(lat0=37.975, lon0=23.725, cell=20.0)
    out = []
    for f in files:
        meta = pneuma.parse_name(f)
        df, dur = pneuma.vehicle_seconds_by_cell(f, grid)
        e = pneuma.road_occupant_exposure(df, dur, grid, pneuma.ExposureAssumptions())
        out.append(e.assign(**meta))
    expo = pd.concat(out); expo.to_parquet('data/processed/pneuma_exposure.parquet') if False else expo.to_csv('data/processed/pneuma_exposure.csv', index=False)

## 4. Solver feasibility on synthetic coefficients

In [ ]:
!python -m scripts.solver_feasibility